# 北海道：第一版の記述用散布図

1点＝30分。全17,520コマ。台帳記載法人に基づく3公開設備の参照群です。
所有・売電権・契約供給力の帰属、実際の稼働可能余力、因果効果は示しません。
[報告を読む](../reports/13_hokkaido_scatter/README.md)。画像はセルを実行しなくても表示できます。


## 主散布図

横軸＝制度対象外公開設備の参照容量差、縦軸＝北海道spot価格、色＝東北→北海道の翌々日策定空容量。
需要群は翌々日の公表予測需要によります。実績需要ではありません。

![主散布図](../reports/13_hokkaido_scatter/figures/hokkaido_reference_gap_scatter.png)


## 補助散布図

価格差の正・ゼロ・負を保持しています。物理的な分断・孤立とは呼びません。

![補助散布図](../reports/13_hokkaido_scatter/figures/hokkaido_import_price_difference.png)


In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

relative = Path("data/processed/description/hokkaido/20261016_scatter_v1")
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / relative).is_dir())
panel = pq.read_table(root / relative / "hokkaido_scatter_panel_FY2025.parquet").to_pandas()
print(f"{len(panel):,}行・{panel.delivery_date.nunique()}日")
display(
    panel[
        [
            "delivery_date",
            "period",
            "R_ref_MW",
            "area_price",
            "forecast_demand_group",
            "import_capacity_direction_display_mw",
            "hokkaido_minus_tohoku_price",
            "non_target_calculated_unit_count",
            "non_target_capacity_reference_mw",
            "non_target_output_missing_count",
            "classification_unknown_unit_count",
        ]
    ].head(20)
)

## 全25設備の分類と参照容量

対象22・対象外3・不明0は、今回の公開観測ラベル内の数です。対象外供給者全体の網羅率ではありません。


In [ ]:
assets = pq.read_table(root / relative / "hokkaido_asset_registry_FY2025.parquet").to_pandas()
display(
    assets[
        [
            "plant_name_raw",
            "unit_name_raw",
            "hjks_business_name_unique_reference",
            "reference_classification",
            "reference_classification_reason",
            "diagnostic_mapping__capacity_reference_raw_kw",
        ]
    ]
)
display(pd.read_csv(root / "reports/13_hokkaido_scatter/qa/forecast_demand_groups.csv"))

## 元列・原値と公表区分の寄与を見る

Parquetには元列を保持しています。公表区分表は4条件を別系列とし、条件間で加算しません。
値の期間和はMW・コマで、エネルギーMWhではありません。


In [ ]:
display(
    pd.DataFrame({"column": panel.columns.astype(str), "dtype": panel.dtypes.astype(str).values})
)
for name in [
    "contribution_fuel_summary.csv",
    "contribution_public_record_summary.csv",
    "asset_set_intervals.csv",
]:
    print(name)
    display(pd.read_csv(root / "reports/13_hokkaido_scatter/qa" / name, keep_default_na=False))